## Exploratory Data Analysis (EDA)

### 1. Trip-Level System Exploration - Bike Share Toronto 
This section explores the raw Bike Share Toronto trip dataset covering two years of system activity. 
Before analyzing operational imbalance, we first explore the behavior of the bike-sharing system at the trip level. This includes usage patterns, temporal demand dynamics, and the distribution of trips across stations.

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

In [0]:
# ==========================================
# Trip-Level EDA (SILVER_TRIPS)
# ==========================================

SILVER_TRIPS = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/bikeshare_trips"

df_trips = spark.read.parquet(SILVER_TRIPS)
print("\n=== Rows ===")
print("Rows (SILVER_TRIPS):", f"{df_trips.count():,}")

# -----------------------------
# Plot style (safe defaults)
# -----------------------------
plt.style.use("default")
sns.set_theme(style="white")


# -----------------------------
# Clean station name fields ("NULL" string / empty -> NULL)
# -----------------------------
df_trips = df_trips.withColumn(
    "start_station_name_clean",
    F.when(F.trim(F.col("start_station_name")) == "NULL", F.lit(None))
     .when(F.trim(F.col("start_station_name")) == "", F.lit(None))
     .otherwise(F.col("start_station_name"))
)

df_trips = df_trips.withColumn(
    "end_station_name_clean",
    F.when(F.trim(F.col("end_station_name")) == "NULL", F.lit(None))
     .when(F.trim(F.col("end_station_name")) == "", F.lit(None))
     .otherwise(F.col("end_station_name"))
)

# -----------------------------
# Schema & Data Types
# -----------------------------
print("\n=== Schema & Data Types ===")
df_trips.printSchema()

# -----------------------------
# General Summary
# -----------------------------
print("\n=== General Summary (describe) ===")
df_trips.describe().display()

print("\n=== Temporal Coverage (year-month-day) ===")
df_trips.select(
    F.min(F.make_date("year", "month", "day")).alias("min_date"),
    F.max(F.make_date("year", "month", "day")).alias("max_date")
).display()

# -----------------------------
# Data Quality Checks
# -----------------------------
print("\n=== Data Quality Checks ===")

#  Trip ID uniqueness (should be unique at trip-level)
dup_trip = df_trips.groupBy("trip_id").count().filter(F.col("count") > 1)
dup_trip_count = dup_trip.count()
print("Duplicate trip_id rows:", dup_trip_count)
if dup_trip_count > 0:
    dup_trip.orderBy(F.desc("count")).limit(20).display()

#  Basic range checks
df_trips_h = df_trips.withColumn(
    "hour_start_int",
    F.regexp_extract(F.col("hour_start_str").cast("string"), r"^(\d{1,2})", 1).cast("int")
)

sanity = df_trips_h.select(
    F.count("*").alias("rows_total"),
    F.count(F.when(~F.col("month").between(1, 12), 1)).alias("bad_month"),
    F.count(F.when(~F.col("day").between(1, 31), 1)).alias("bad_day"),
    F.count(F.when(~F.col("hour_start_int").between(0, 23), 1)).alias("bad_hour_start"),
    F.count(F.when(F.col("trip_duration_min").isNull(), 1)).alias("null_trip_duration"),
    F.count(F.when(F.col("trip_duration_min") <= 0, 1)).alias("nonpositive_duration")
)
sanity.display()

# -----------------------------
# IQR Outlier Detection (trip_duration_min)
# -----------------------------
quantiles = df_trips.approxQuantile("trip_duration_min", [0.25, 0.75], 0.01)
Q1, Q3 = quantiles[0], quantiles[1]
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print("\n=== IQR Outlier Detection (trip_duration_min) ===")
print("Q1:", Q1)
print("Q3:", Q3)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

df_outliers = df_trips.filter(
    (F.col("trip_duration_min") < lower_bound) |
    (F.col("trip_duration_min") > upper_bound)
)

print("Total Outliers:", df_outliers.count())

# -----------------------------
# Trip Duration Boxplot (Log Scale)
# -----------------------------
pdf = df_trips.select("trip_duration_min").toPandas()
pdf["log_trip_duration"] = np.log1p(pdf["trip_duration_min"])

plt.figure(figsize=(6, 4))
sns.boxplot(y=pdf["log_trip_duration"], fliersize=3)
plt.title("Trip Duration Distribution (Log Scale)", fontsize=15, fontweight='bold', pad=20)
plt.ylabel("log(Trip Duration Minutes + 1)")
plt.xlabel("")
plt.tight_layout()
plt.show()

# -----------------------------
# Monthly Ridership Trend (Full Period)
# -----------------------------
monthly = (
    df_trips
    .groupBy("year", "month")
    .count()
    .orderBy("year", "month")
    .toPandas()
)

monthly["year_month"] = (
    monthly["year"].astype(str) + "-" +
    monthly["month"].astype(str).str.zfill(2)
)

plt.figure(figsize=(10, 6))

plt.plot(
    monthly["year_month"],
    monthly["count"],
    marker="o"
)

# Add labels above each point
for i, value in enumerate(monthly["count"]):
    plt.text(
        i,
        value,
        f"{int(value):,}",
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.xticks(rotation=45)
plt.title("Monthly Ridership Trend (Full Period)", fontsize=15, fontweight='bold', pad=20)
plt.ylabel("Trip Count")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# -----------------------------
# Top 10 Start Stations (avoid 'NULL')
# -----------------------------
print("\n=== Top 10 Start Stations (by trips) ===")

top_stations_df = (
    df_trips
    .filter(F.col("start_station_name_clean").isNotNull())
    .groupBy("start_station_name_clean")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

top_stations_pd = top_stations_df.toPandas()

plt.figure(figsize=(10, 6))

plt.barh(
    top_stations_pd["start_station_name_clean"],
    top_stations_pd["count"]
)

plt.xlabel("Trip Count")
plt.ylabel("Station Name")
plt.title("Top 10 Start Stations (All Data)", fontsize=15, fontweight='bold', pad=20)
plt.gca().invert_yaxis()

# Add labels
for i, v in enumerate(top_stations_pd["count"]):
    plt.text(
        v,
        i,
        f" {int(v):,}",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

# -----------------------------
# Trips by Hour of Day (hour_start_str)
# -----------------------------
hourly = (
    df_trips
    .withColumn("hour", F.substring("hour_start_str", 1, 2).cast("int"))
    .groupBy("hour")
    .count()
    .orderBy("hour")
    .toPandas()
)

plt.figure(figsize=(10, 6))

bars = plt.bar(hourly["hour"], hourly["count"])

plt.title("Trips by Hour of Day (2 Years)", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Hour")
plt.ylabel("Trip Count")

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{int(height):,}",
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.xticks(hourly["hour"])
plt.grid(True, axis="y", linestyle="--", alpha=0.25)
plt.tight_layout()
plt.show()

# -----------------------------
# Weekly Usage Behavior (Trips by Day of Week)
# -----------------------------
df_week = df_trips.withColumn(
    "date",
    F.to_date(F.concat_ws("-", "year", "month", "day"))
).withColumn("day_num", F.dayofweek("date")) \
 .withColumn("weekday", F.date_format("date", "E"))

weekly = (
    df_week
    .groupBy("day_num", "weekday")
    .count()
    .orderBy("day_num")
    .toPandas()
)

plt.figure(figsize=(10, 6))

bars = plt.bar(weekly["weekday"], weekly["count"], width=0.4)

plt.title("Trips by Day of Week (2 Years)", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Day of Week")
plt.ylabel("Trip Count")

# Add labels
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{int(height):,}",
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.grid(True, axis="y", linestyle="--", alpha=0.25)
plt.tight_layout()
plt.show()


Trip-level analysis reveals clear temporal demand patterns across the system. The Monthly Ridership, Trips by Hour, and Trips by Day of Week visualizations highlight strong temporal dynamics in system usage, while the Top 10 Start Stations chart shows a high concentration of trips in a small subset of stations. The Trip Duration boxplot also reveals a right-skewed distribution with outliers, indicating variability in user trip behavior across the network.

### 2. Operational Flow Analysis Bike Share Toronto (Station-Hour Aggregation)
In this section, the analysis moves from individual trips to an operational perspective by using the Bike Share aggregated dataset. Trips are grouped at the station-hour level, allowing the computation of key operational metrics such as departures, arrivals, and net flow. This aggregation provides a clearer view of system dynamics and enables the identification of temporal patterns and station-level imbalances that are not observable at the trip level, highlighting locations where bike redistribution pressure is more likely to occur.

In [0]:
# ============================================================
# Operational EDA (SILVER_AGG_DIR: station-hour flow)
# ============================================================

SILVER_AGG_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver_agg/station_hour_flow"
df_flow = spark.read.parquet(SILVER_AGG_DIR)

print("Rows:", f"{df_flow.count():,}")

print("Schema:")
df_flow.printSchema()

# ============================================================
# General Summary (key metrics)
# ============================================================
print("\n=== General Summary (Operational) ===")
df_flow.select("departures", "arrivals", "net_flow").describe().show()

# ============================================================
# Missing Values Count (all columns)
# ============================================================
print("\n=== Missing Values Count (all columns) ===")
missing = df_flow.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df_flow.columns
])
display(missing)

# ============================================================
# Data Quality Checks (grain uniqueness + ranges)
# ============================================================
print("\n=== Data Quality Checks ===")
grain = ["station_id","year","month","day","hour"]

dups = df_flow.groupBy(*grain).count().filter(F.col("count") > 1)
dup_count = dups.count()
print("Duplicate grain rows:", dup_count)
if dup_count > 0:
    display(dups.orderBy(F.desc("count")).limit(50))

bad_ranges = df_flow.filter(
    (~F.col("hour").between(0,23)) |
    (~F.col("month").between(1,12)) |
    (~F.col("day").between(1,31)) |
    (F.col("departures") < 0) |
    (F.col("arrivals") < 0)
)
bad_count = bad_ranges.count()
print("Bad range rows:", bad_count)
if bad_count > 0:
    display(bad_ranges.select(*grain, "departures","arrivals","net_flow").limit(50))

# ============================================================
# Monthly Activity Trend 
# ============================================================
monthly = (
    df_flow
    .groupBy("year", "month")
    .agg(
        (F.sum("departures") + F.sum("arrivals")).alias("total_trips")
    )
    .orderBy("year", "month")
    .toPandas()
)

monthly["year_month"] = monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)

fig, ax1 = plt.subplots(figsize=(11, 6))

ax1.plot(
    monthly["year_month"],
    monthly["total_trips"],
    marker="o",
    color="blue",
    label="Total Trips"
)

ax1.set_xlabel("Year-Month")
ax1.set_ylabel("Total Trips", color="blue")
ax1.tick_params(axis="y", labelcolor="blue")
ax1.grid(True)

plt.xticks(rotation=45)
plt.title("Monthly Volume", fontsize=15, fontweight="bold", pad=20)
fig.tight_layout()
plt.show()

# ============================================================
# Hour-of-Day Pattern (departures/arrivals)
# ============================================================
hourly = (
    df_flow
    .groupBy("hour")
    .agg(
        F.sum("departures").alias("departures_sum"),
        F.sum("arrivals").alias("arrivals_sum"),
    )
    .orderBy("hour")
    .toPandas()
)

plt.figure(figsize=(10,6))

plt.plot(hourly["hour"], hourly["departures_sum"],
         marker="o", linewidth=2, label="Departures", color="blue")

plt.plot(hourly["hour"], hourly["arrivals_sum"],
         marker="o", linewidth=2, label="Arrivals", color="orange")

plt.title("Trips by Hour of Day (Operational)", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Hour")
plt.ylabel("Trips")
plt.xticks(hourly["hour"])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# Top 15 Stations by Departures
# ============================================================

top_dep = (
    df_flow
    .groupBy("station_id")
    .agg(F.sum("departures").alias("departures_sum"))
    .orderBy(F.desc("departures_sum"))
    .limit(15)
    .toPandas()
)

plt.figure(figsize=(10, 6))
plt.barh(top_dep["station_id"].astype(str), top_dep["departures_sum"], color="steelblue")
plt.gca().invert_yaxis()
plt.title("Top 15 Stations by Total Departures", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Total Departures")
plt.ylabel("Station ID")

for i, v in enumerate(top_dep["departures_sum"]):
    plt.text(v, i, f"{int(v):,}", va="center", ha="left", fontsize=9)

plt.tight_layout()
plt.show()

# ============================================================
# Top 15 Stations by Arrivals
# ============================================================

top_arr = (
    df_flow
    .groupBy("station_id")
    .agg(F.sum("arrivals").alias("arrivals_sum"))
    .orderBy(F.desc("arrivals_sum"))
    .limit(15)
    .toPandas()
)

plt.figure(figsize=(10, 6))
plt.barh(top_arr["station_id"].astype(str), top_arr["arrivals_sum"], color="lightblue")
plt.gca().invert_yaxis()
plt.title("Top 15 Stations by Total Arrivals", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Total Arrivals")
plt.ylabel("Station ID")

for i, v in enumerate(top_arr["arrivals_sum"]):
    plt.text(v, i, f"{int(v):,}", va="center", ha="left", fontsize=9)

plt.tight_layout()
plt.show()

# ============================================================
# Top 15 Stations by Imbalance Intensity
# ============================================================

top_imb = (
    df_flow
    .groupBy("station_id")
    .agg(
        F.sum(F.abs(F.col("net_flow"))).alias("sum_abs_net_flow"),
        F.avg(F.abs(F.col("net_flow"))).alias("avg_abs_net_flow")
    )
    .orderBy(F.desc("sum_abs_net_flow"))
    .limit(15)
    .toPandas()
)

plt.figure(figsize=(10, 6))
plt.barh(top_imb["station_id"].astype(str), top_imb["sum_abs_net_flow"], color="dodgerblue")
plt.gca().invert_yaxis()
plt.title("Top 15 Stations by Imbalance Intensity (Σ |net_flow|)", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Sum of |net_flow|")
plt.ylabel("Station ID")

for i, v in enumerate(top_imb["sum_abs_net_flow"]):
    plt.text(v, i, f"{int(v):,}", va="center", ha="left", fontsize=9)

plt.tight_layout()
plt.show()

# ============================================================
# Net Flow Outliers (Boxplot) Log scale on |net_flow|
# ============================================================
# Log scale of absolute net_flow (helps show tails)
net_pdf = df_flow.select("net_flow").toPandas()
net_pdf["abs_net_flow"] = net_pdf["net_flow"].abs()
net_pdf["log_abs_net_flow"] = np.log1p(net_pdf["abs_net_flow"])

# Log scale of absolute net_flow 
plt.figure(figsize=(10,6))
sns.boxplot(y=net_pdf["log_abs_net_flow"], fliersize=3)
plt.title("Absolute Net Flow Outliers (Log Scale)", fontsize=15, fontweight='bold', pad=20)
plt.ylabel("log(|net_flow| + 1)")
plt.tight_layout()
plt.show()


Operational aggregation reveals clear temporal cycles in demand as well as differences between arrivals and departures across stations and times of day. These patterns highlight how spatial and temporal fluctuations in demand can generate operational imbalances within the system. The identification of stations with higher imbalance intensity further emphasizes the importance of analyzing net flow dynamics before incorporating external contextual factors such as weather and events.

### 3. Enriched System Analysis (Integrated Dataset: GOLD_V2)
This section introduces the integrated analytical dataset used for the subsequent stages of the exploratory analysis. The GOLD_V2 dataset combines station-hour operational metrics with contextual variables including weather conditions, station metadata, and large-scale public events. This enriched dataset enables a deeper investigation of the factors influencing bike-share demand and station imbalance, allowing the analysis to move beyond basic operational patterns and examine how temporal, environmental, and spatial factors affect system dynamics.

In [0]:
# ============================================================
# Contextual / Integrated EDA (GOLD_V2)
# # ============================================================

GOLD_V2 = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/gold/gold_v2_spatiotemporal_events"

df_gold = spark.read.parquet(GOLD_V2)

print("Rows (GOLD_V2):", f"{df_gold.count():,}")
print("Schema:")
df_gold.printSchema()

# -----------------------------
# Basic temporal coverage
# -----------------------------
print("\n=== Temporal Coverage ===")
(df_gold.select(
    F.min(F.make_date("year", "month", "day")).alias("min_date"),
    F.max(F.make_date("year", "month", "day")).alias("max_date"),
).display())

# -----------------------------
# Core distributions (net_flow + abs_net_flow)
# -----------------------------
print("\n=== Net flow summary ===")
df_gold.select("net_flow").describe().display()

print("\n=== Absolute net flow (imbalance magnitude) summary ===")
df_gold.select(F.abs("net_flow").alias("abs_net_flow")).describe().display()

print("\n=== Net flow distribution (zoomed: -50..50) ===")
net_pdf = (
    df_gold.select("net_flow")
    .filter(F.col("net_flow").between(-50, 50))
    .toPandas()
)

plt.figure(figsize=(10,6))
sns.histplot(net_pdf["net_flow"], bins=101, kde=True)
plt.title("Net Flow Distribution (Zoomed: -50 to +50)", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("net_flow (arrivals - departures)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# -----------------------------
# Hourly Imbalance Severity (Weekday vs Weekend)
# -----------------------------

# Create timestamp + helper columns
df_gold = df_gold.withColumn(
    "ts",
    F.to_timestamp(
        F.concat_ws(
            " ",
            F.concat_ws("-", F.col("year"), F.lpad(F.col("month"), 2, "0"), F.lpad(F.col("day"), 2, "0")),
            F.concat(F.lpad(F.col("hour"), 2, "0"), F.lit(":00:00"))
        ),
        "yyyy-MM-dd HH:mm:ss"
    )
).withColumn("date", F.to_date("ts")) \
 .withColumn("year_month", F.date_format("ts", "yyyy-MM")) \
 .withColumn("dayofweek", F.dayofweek("ts")) \
 .withColumn("is_weekend", F.col("dayofweek").isin([1, 7]).cast("int")) \
 .withColumn("hour_of_day", F.hour("ts"))


# Demand and imbalance magnitude proxies
if "departures" in df_gold.columns and "arrivals" in df_gold.columns:
    df_gold = df_gold.withColumn("total_demand", F.col("departures") + F.col("arrivals"))
else:
    df_gold = df_gold.withColumn("total_demand", F.lit(None).cast("double"))

if "net_flow" in df_gold.columns:
    df_gold = df_gold.withColumn("abs_net_flow", F.abs(F.col("net_flow")))
else:
    df_gold = df_gold.withColumn("abs_net_flow", F.lit(None).cast("double"))
    
# Hourly severity (weekday vs weekend)
hourly_abs = (
    df_gold.groupBy("hour_of_day", "is_weekend")
        .agg(F.avg("abs_net_flow").alias("avg_abs_net_flow"))
        .orderBy("hour_of_day")
)
pdf_hour_abs = hourly_abs.toPandas()
wk_hour = pdf_hour_abs[pdf_hour_abs["is_weekend"] == 0]
we_hour = pdf_hour_abs[pdf_hour_abs["is_weekend"] == 1]

# 2) Weekly severity
weekly_abs = (
    df_gold.groupBy("dayofweek")
        .agg(F.avg("abs_net_flow").alias("avg_abs_net_flow"))
        .orderBy("dayofweek")
)
pdf_week_abs = weekly_abs.toPandas()

# 3) Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Hourly severity
axes[0].plot(wk_hour["hour_of_day"], wk_hour["avg_abs_net_flow"], marker="o", label="Weekday")
axes[0].plot(we_hour["hour_of_day"], we_hour["avg_abs_net_flow"], marker="o", linestyle="--", label="Weekend")
axes[0].set_title("Hourly Imbalance Severity (abs_net_flow)")
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Average |arrivals − departures|")
axes[0].set_xticks(range(0, 24))
axes[0].grid(alpha=0.2)
axes[0].legend()

# Right: Weekly severity
axes[1].plot(pdf_week_abs["dayofweek"], pdf_week_abs["avg_abs_net_flow"], marker="o")
axes[1].set_title("Weekly Imbalance Severity (abs_net_flow)")
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Average |arrivals − departures|")
axes[1].set_xticks([1,2,3,4,5,6,7])
axes[1].set_xticklabels(["Sun","Mon","Tue","Wed","Thu","Fri","Sat"])
axes[1].grid(alpha=0.2)

fig.suptitle("Temporal Severity of Station Imbalance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()



# ------------------------------
# Station-Level Behavioral Patterns
# ------------------------------

# Station risk table 

station_risk = (
    df_gold.groupBy("station_id")
        .agg(
            F.avg("abs_net_flow").alias("mean_abs_net_flow"),
            F.count("*").alias("n_obs")
        )
)

pdf_risk = station_risk.toPandas().sort_values("mean_abs_net_flow", ascending=False)


# Risk concentration (Top 10%)

top_frac = 0.10
k = max(1, int(len(pdf_risk) * top_frac))

risk_share = pdf_risk.head(k)["mean_abs_net_flow"].sum() / pdf_risk["mean_abs_net_flow"].sum()

# print(f"Risk concentration: Top {int(top_frac*100)}% stations explain {risk_share:.2%} of total imbalance severity.\n")



# Visual: Top N bar chart (compact)
topN = 15
plt.figure(figsize=(10, 6))
plt.bar(pdf_risk.head(topN)["station_id"].astype(str), pdf_risk.head(topN)["mean_abs_net_flow"])
plt.title(f"Top {topN} High-Risk Stations — Mean abs_net_flow", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Station ID")
plt.ylabel("Mean abs_net_flow")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

# Top 15 stations ranked by average imbalance severity (mean |net_flow|). Highlights structurally high-risk locations with persistent redistribution pressure.

plt.show()

# ------------------------------
# 9) Extreme Imbalance Analysis
# -----------------------------

# Compute thresholds

quantiles = df_gold.approxQuantile("abs_net_flow", [0.90, 0.95], 0.01)

p90 = quantiles[0]
p95 = quantiles[1]

#print(f"p90 threshold: {p90:.2f}")
print(f"p95 threshold: {p95:.2f}")


# Create extreme imbalance flag

df_extreme = df_gold.withColumn(
    "extreme_imbalance_flag",
    F.when(F.col("abs_net_flow") >= p95, 1).otherwise(0)
)

# Overall frequency

extreme_rate = (
    df_extreme.agg(F.avg("extreme_imbalance_flag")).collect()[0][0]
)

print(f"\nOverall extreme imbalance rate (p95): {extreme_rate:.2%}")


# Event vs Non-Event comparison

event_comparison = (
    df_extreme.groupBy("event_active_nearby_flag")
        .agg(F.avg("extreme_imbalance_flag").alias("extreme_rate"))
        .orderBy("event_active_nearby_flag")
)

event_comparison.show()


# Hourly pattern of extreme risk

hourly_extreme = (
    df_extreme.groupBy("hour")
        .agg(F.avg("extreme_imbalance_flag").alias("extreme_rate"))
        .orderBy("hour")
)

pdf_hour_extreme = hourly_extreme.toPandas()

plt.figure(figsize=(10, 6))
plt.plot(pdf_hour_extreme["hour"], pdf_hour_extreme["extreme_rate"])
plt.title("Hourly Extreme Imbalance Risk (p95)", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Hour of Day")
plt.ylabel("Probability of Extreme Imbalance")
plt.tight_layout()

plt.show()
#Probability of extreme imbalance (p95 threshold) by hour of day. Reveals concentration of critical risk during peak commuting periods and amplification under event conditions.

# -----------------------------
# Correlation Matrix
# -----------------------------
cols = [
    "total_demand",
    "abs_net_flow",
    "net_flow",
    "hour",
    "is_weekend",
    "temperature_2m_celsius",
    "apparent_temperature_celsius",
    "event_active_nearby_flag",
    "event_weighted_intensity",
    "event_impact_score"
]

cols = [c for c in cols if c in df_gold.columns]

df_corr = df_gold.select([F.col(c).cast("double").alias(c) for c in cols])

corr_rows = []
for c1 in cols:
    row = {"var": c1}
    for c2 in cols:
        row[c2] = df_corr.stat.corr(c1, c2)
    corr_rows.append(row)

corr_pdf = pd.DataFrame(corr_rows).set_index("var")

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(
    corr_pdf.values,
    cmap="Blues",      
    vmin=-1,
    vmax=1
)

ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticklabels(cols)

for i in range(len(cols)):
    for j in range(len(cols)):
        val = corr_pdf.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax)
plt.title("Correlation Matrix — GOLD_V2 ", fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

Analysis of the integrated GOLD_V2 dataset reveals clear structural patterns in station imbalance across both temporal and spatial dimensions. Net flow distributions show that most station-hour observations remain close to balance, while a subset of stations and time periods exhibit significantly higher imbalance severity. 

Temporal analysis highlights recurring hourly and weekly imbalance patterns, and station-level risk metrics indicate that a relatively small group of stations accounts for a disproportionate share of redistribution pressure. 

Extreme imbalance events, although infrequent, tend to concentrate during specific operational periods, motivating further investigation of contextual drivers such as weather conditions and public events.

### 4. Weather Effects on Bike Share Demand and Imbalance
Weather conditions can significantly influence bike-sharing usage patterns by affecting rider comfort, safety, and travel preferences. This section explores how temperature, precipitation, and wind conditions relate to demand levels and imbalance severity across the Bike Share Toronto network.

##### Temperature vs Net Flow by Nearby Event Activity 
Scatter plot showing the relationship between temperature and bike net flow (arrivals minus departures), colored by whether a nearby event was active. This helps identify whether temperature influences demand differently when events are occurring nearby.

In [0]:
GOLD_V2_PATH = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/gold/gold_v2_spatiotemporal_events"
df_v2 = spark.read.parquet(GOLD_V2_PATH)

# Select relevant columns: weather, net_flow, events (V1 daily + V2 spatiotemporal)
core_cols = [
    "year", "month", "day", "hour",
    "temperature_2m_celsius", "apparent_temperature_celsius",
    "net_flow"
]

v2_event_cols = [
    "event_day_flag", "event_day_attendance_sum", "events_day_count",
    "event_active_nearby_flag", "event_attendance_est_sum_nearby",
    "events_nearby_count", "event_weighted_intensity",
    "event_impact_category", "event_impact_score"
]

df_v2_eda = df_v2.select(*(core_cols + v2_event_cols))

print("\nSelected columns for EDA:")
print(f"V2: {df_v2_eda.columns}")

In [0]:
# ------------------------------
# Weather — Temperature vs Net Flow by Nearby Event Flag
# ------------------------------

# Sample for plotting
pdf_v2 = df_v2_eda.sample(fraction=0.05, seed=42).toPandas()

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=pdf_v2,
    x="temperature_2m_celsius",
    y="net_flow",
    hue="event_active_nearby_flag",
    palette=["steelblue", "red"],
    alpha=0.5,
    s=12
   
)

ax.set_title("Temperature vs Net Flow by Nearby Event Flag", fontsize=15, fontweight='bold', pad=20)
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Net Flow (arrivals - departures)")
ax.set_ylim(-50, 50)  # zoom for clarity (net_flow outliers)
plt.tight_layout()
plt.show()


- Net flow variability increases under moderate temperature conditions, suggesting higher bike-sharing activity when weather conditions are comfortable. While extreme temperatures appear to reduce system usage, nearby events amplify variability in net flow, indicating that favorable weather combined with event-driven demand can intensify station-level imbalance.


##### Average Bike Share Demand by Temperature and Event Activity (Heatmap)

In [0]:
df_heat = (
    df_gold
    .withColumn(
        "temp_bin",
        F.when(F.col("temperature_2m_celsius") < 0, "Cold")
         .when(F.col("temperature_2m_celsius") < 15, "Mild")
         .otherwise("Warm")
    )
    .groupBy("temp_bin", "event_active_nearby_flag")
    .agg(F.avg("total_demand").alias("avg_demand"))
    .toPandas()
)

pivot = df_heat.pivot(
    index="temp_bin",
    columns="event_active_nearby_flag",
    values="avg_demand"
)

sns.heatmap(pivot, annot=True, cmap="Blues")
plt.title("Bike Share Demand by Temperature and Event Activity", fontsize=15, fontweight='bold', pad=20)

- Bike-share demand increases noticeably under warmer weather conditions, with the highest average demand observed when warm temperatures coincide with nearby public events. This suggests that comfortable weather combined with event-driven activity significantly amplifies system usage, potentially increasing operational pressure and imbalance at nearby stations.

##### Net Flow Distribution by Temperature Bin and Event Impact
Boxplot comparing net bike flow across temperature categories (Cold, Mild, Warm) and event impact levels. This reveals how the combination of weather conditions and event size jointly affects bike station demand.

In [0]:
# Aggregations: Net flow by weather bins, event flags, etc.
# Create temperature bins for grouping (e.g., cold, mild, warm)
df_v2_eda = df_v2_eda.withColumn(
    "temp_bin",
    F.when(F.col("temperature_2m_celsius") < 0, "Cold (<0°C)")
     .when(F.col("temperature_2m_celsius") < 15, "Mild (0-15°C)")
     .otherwise("Warm (>15°C)")
)

# Sample to Pandas for visualizations
print("\nSampling ~57,000 rows for plotting...")

pdf_v2 = df_v2_eda.sample(fraction=0.05, seed=42).toPandas()

print(f"Sampled rows V2: {len(pdf_v2):,}")

# Add temp_bin to Pandas for consistency
pdf_v2["temp_bin"] = pdf_v2["temperature_2m_celsius"].apply(
    lambda x: "Cold (<0°C)" if x < 0 else "Mild (0-15°C)" if x < 15 else "Warm (>15°C)"
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("coolwarm")

# ------------------------------
# Net Flow by Temperature Bin and Event Impact Category
# ------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=pdf_v2, x="temp_bin", y="net_flow", hue="event_impact_category", ax=ax)
ax.set_title("Net Flow Distribution by Temperature Bin and Event Impact (V2 sample)", fontsize=15, fontweight='bold', pad=20)
ax.set_ylim(-20, 20)  # Focus on central range
plt.tight_layout()
plt.show()

- Net flow distributions vary across temperature ranges, with warmer conditions generally associated with greater variability in station-level flows. This suggests that comfortable weather conditions increase overall activity in the system, which may lead to higher imbalance intensity during periods of elevated demand.

### 5. EDA for the impact of public events in Toronto Bike-Share


##### 5.1 Analysis by event distribution

##### Hourly Net Flow by Event Day 

Line chart showing how average bike net flow changes throughout the day, split by whether it is an event day (V1). This makes it easy to compare hourly demand patterns on event days versus non-event days.

In [0]:
agg_hour_v2 = (
    df_v2_eda
    .groupBy("hour", "event_day_flag")
    .agg(F.avg("net_flow").alias("avg_net_flow"))
    .orderBy("hour", "event_day_flag")
    .toPandas()
)
fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(data=agg_hour_v2, x="hour", y="avg_net_flow", hue="event_day_flag", ax=ax)
ax.set_title("Hourly Net Flow by Event Day", fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

- Hourly net flow patterns reveal clear temporal dynamics, with stronger directional flows during specific hours of the day. Differences between event and non-event days suggest that external activities may amplify typical mobility patterns, potentially increasing imbalance risk during peak activity periods.

Event Impact Ratio:
- It analyzes how many records correspond to each impact category

In [0]:
path = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/gold/gold_v2_spatiotemporal_events"
df_goldv2 = spark.read.parquet(path)

distribution_impact = df_goldv2.groupBy("event_impact_category").count() \
    .withColumn("percentage", (F.col("count") / df_goldv2.count()) * 100) \
    .orderBy(F.desc("percentage"))

print("Distribution by Impact Category:")
display(distribution_impact)

- Public events are extremely rare in the dataset. Nearly 99% of station-hour observations occur without nearby events, indicating that event-related demand shocks represent a very small but potentially impactful subset of system activity.

The dataset is overwhelmingly dominated by “No Event” entries (≈99%), meaning all other impact categories combined make up less than 1% of the data.

Top 10 Event Categories

In [0]:
top_10_df = df_goldv2.select(F.explode("event_day_categories").alias("category")) \
    .groupBy("category") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .toPandas()


plt.figure(figsize=(10, 6))
sns.set_style("white")
ax = sns.barplot(data=top_10_df, x='count', y='category', color="royalblue")


for p in ax.patches:
    width = p.get_width()
    ax.text(width + 5000, # space next to the bar
            p.get_y() + p.get_height()/2, 
            f'{int(width):,}', # thounsands separator
            va='center', fontsize=11, fontweight='bold')


plt.title("Top 10 Event Day Categories in Toronto", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("# of Records", fontsize=12)
plt.ylabel("Category", fontsize=12)
sns.despine(trim=True)

plt.tight_layout()
plt.show()

Event Category Distribution in Toronto

- Strong dominance of Arts as the leading category.
- High volume also in Sports and Festival.
- Clear drop after the top three groups.

Toronto’s event landscape is heavily driven by cultural and sports activities, revealing a city where artistic expression, major sporting events, and large festivals shape most public engagement and urban activity patterns

Average imbalance according to event distance:

In [0]:
df_distance = df_goldv2.filter(F.col("nearest_event_km") < 5) \
    .withColumn("distance", 
        F.when(F.col("nearest_event_km") < 0.5, "1. Very close (<500m)")
        .when(F.col("nearest_event_km") < 1.0, "2. close (500m-1km)")
        .otherwise("3. Area of ​​Influence (1km-5km)"))


proximity_analysis = df_distance.groupBy("distance") \
    .agg(
        F.avg(F.abs("net_flow")).alias("avg_desbalance"),
        F.count("*").alias("n_records")
    ).orderBy("distance")

print("Imbalance according to the distance: ")
display(proximity_analysis)

##### 5.2 Analysis by Event Name

In [0]:
# 1. Filtering by distance (< 1km) and explode the names

df_event_names = df_goldv2.filter(F.col("nearest_event_km") <= 1.0) \
    .withColumn("name_of_event", F.explode("event_names_nearby"))

top_events_impact = df_event_names.groupBy("name_of_event") \
    .agg(
        F.avg(F.abs("net_flow")).alias("imbalance_net_flow_avg"),
        F.max(F.abs("net_flow")).alias("imbalance_net_flow_max"),
        F.count("*").alias("count")
    ) \
    .filter(F.col("count") > 10) \
    .orderBy(F.desc("imbalance_net_flow_avg"))

# convert Top 10 
pdf_top_eventos = top_events_impact.limit(10).toPandas()

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

ax = sns.barplot(data=pdf_top_eventos, x='imbalance_net_flow_avg', y='name_of_event', color="royalblue")

for i, p in enumerate(ax.patches):
    avg_val = pdf_top_eventos.iloc[i]['imbalance_net_flow_avg']
    max_val = pdf_top_eventos.iloc[i]['imbalance_net_flow_max']
    
    ax.text(avg_val + 0.2, p.get_y() + p.get_height()/2, 
            f'{avg_val:.2f}', 
            va='center', fontsize=11, fontweight='bold', color='#444')

# 5. Personalización final
plt.title("Top 10 Events by Imbalance Impact", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Average Absolute Imbalance (Imbalance Net Flow Abs)", fontsize=12)
plt.ylabel("Event Name", fontsize=12)
plt.xlim(0, pdf_top_eventos['imbalance_net_flow_avg'].max() + 5) 

sns.despine()
plt.tight_layout()
plt.show()

- Clear standout impact from Woodbine Beach Fireworks.
- Concerts at Budweiser Stage show consistently high imbalance.
- Film festivals and major markets generate strong but secondary effects.

Crowd-flow pressure concentrates heavily in a few marquee events, especially fireworks and large concerts, signaling where Toronto’s mobility systems face their sharpest stress and where planning interventions matter most.


In [0]:
# Extract the Top 10 event names dynamically without using RDDs
top_10_rows = top_events_impact.limit(10).select("name_of_event").collect()
top_10_events_list = [row["name_of_event"] for row in top_10_rows]

print(f"Analyzing impact for the following top 10 events: {top_10_events_list}")

# Tag records that belong to any of these 10 events
df_dynamic_top = df_event_names.withColumn(
    "is_top_event", 
    F.col("name_of_event").isin(top_10_events_list)
)

# Aggregate by hour and event category
chaos_curve_raw = df_dynamic_top.groupBy("hour", "is_top_event") \
    .agg(F.avg(F.abs("net_flow")).alias("avg_imbalance")) \
    .orderBy("hour") \
    .toPandas()

# Pivot the data to ensure both series are aligned for plotting
chaos_pivot = chaos_curve_raw.pivot(index='hour', columns='is_top_event', values='avg_imbalance').fillna(0)
chaos_pivot.columns = ['Other_Events' if col == False else 'Top_10_Events' for col in chaos_pivot.columns]
chaos_pivot = chaos_pivot.reset_index()

plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

plt.plot(chaos_pivot['hour'], chaos_pivot['Top_10_Events'], 
         label='Top 10 High-Impact Events', color='#e74c3c', marker='o', linewidth=3.5)
plt.plot(chaos_pivot['hour'], chaos_pivot['Other_Events'], 
         label='Baseline (Other Events)', color='blue', marker='o', linewidth=2, alpha=0.7)

plt.fill_between(chaos_pivot['hour'], 
                 chaos_pivot['Top_10_Events'], 
                 chaos_pivot['Other_Events'],
                 where=(chaos_pivot['Top_10_Events'] > chaos_pivot['Other_Events']),
                 alpha=0.2, color='red', label='Impact Gap')

plt.title("Imbalance Dynamics: Top Events vs Baseline (Chaos Curve)", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Hour of Day (24h)", fontsize=12)
plt.ylabel("Average Absolute Imbalance (Net Flow)", fontsize=12)
plt.xticks(range(0, 24))
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

This chart shows one important thing:  how human movement behaves differently during the top 10 high‑impact events compared to all other events, hour by hour.

- Top events create sharper, more volatile peaks. This means:
  - Movement is more synchronized (everyone leaves at once).
  - Surges are more directional (large net flow imbalance).
  - Operational stress is concentrated in short windows.

- Baseline events produce smoother, flatter curves. It means:
  - Distributed arrivals and departures.
  - Lower synchronization.
  - Less extreme directional flow

### 6. Key Findings from the Exploratory Analysis

The exploratory analysis reveals several structural patterns in the Bike Share Toronto system.

First, trip-level analysis shows strong temporal demand cycles, with clear peaks during commuting hours and a high concentration of trips among a relatively small subset of stations.

Second, operational flow analysis highlights persistent station-level imbalances, where departures and arrivals diverge across locations and times of day. A small group of stations accounts for a disproportionate share of redistribution pressure.

Third, the integrated GOLD_V2 dataset reveals that imbalance severity is not purely operational but is influenced by contextual factors. Weather conditions affect overall activity levels, with moderate temperatures associated with higher system usage and greater variability in net flow.

Finally, public events introduce localized demand shocks. While events are relatively rare in the dataset, high-impact events generate sharper and more synchronized mobility patterns, concentrating directional flows within short time windows and increasing operational stress around specific venues.

These findings suggest that accurate prediction of station imbalance requires models that incorporate both operational patterns and contextual drivers such as weather conditions and public events.

This exploratory analysis revealed that bike-share demand and station imbalance are  influenced by temporal patterns, weather conditions, and localized public events. These findings motivate the next stage of the project: developing predictive models to forecast arrivals, departures, and station imbalance risk.